用于数据分析

In [1]:
import sys
import os

# 找到项目的根目录（b.ipynb 的上一级目录）
project_root = os.path.abspath("../../")
sys.path.append(project_root)

In [2]:
import pandas as pd

In [8]:
file_path = '/home/kunama/Personal/Stock/Code/trade/data_center/storage/market_data/stock_daily/000001.SZ.parquet'

In [9]:
df = pd.read_parquet(file_path)

In [11]:
from pathlib import Path

DATA_DIR = Path("/home/kunama/Personal/Stock/Code/trade/data_center/storage/market_data/stock_daily/")

def main():
    records = []

    files = sorted(DATA_DIR.glob("*.parquet"))
    print(f"发现 parquet 文件数量: {len(files)}")

    for fp in files:
        try:
            df = pd.read_parquet(fp)

            # 如果 index 就是 datetime（你贴出来的就是这种形式），可以直接用：
            if not isinstance(df.index, pd.DatetimeIndex):
                # 保险一点：如果不是 DatetimeIndex，就尝试把 'datetime' 列转成 index
                if "datetime" in df.columns:
                    df["datetime"] = pd.to_datetime(df["datetime"])
                    df = df.set_index("datetime")
                else:
                    print(f"⚠️ {fp.name} 没有 DatetimeIndex 也没有 'datetime' 列，跳过")
                    continue

            # 按时间排序，确保第一行是最早，最后一行是最晚
            df = df.sort_index()

            start_dt = df.index[0]
            end_dt = df.index[-1]
            n_rows = len(df)

            # code 可以从列里取，也可以从文件名解析
            code_in_df = df["code"].iloc[0] if "code" in df.columns else fp.stem

            records.append({
                "file": fp.name,
                "code": code_in_df,
                "start": start_dt,
                "end": end_dt,
                "rows": n_rows,
            })

        except Exception as e:
            print(f"❌ 读取 {fp} 失败: {e}")
            continue

    if not records:
        print("没有成功统计到任何文件")
        return

    summary = pd.DataFrame(records)
    # 按代码排序，也可以按 start 排
    summary = summary.sort_values(by=["code", "start"])

    # 打印前几行看看
    print("\n=== 每个标的的起止日期 & 行数（前 10 条） ===")
    print(summary.head(10))

    # 保存一份汇总表
    out_file = DATA_DIR / "stock_daily_datetime_range_summary.csv"
    summary.to_csv(out_file, index=False, encoding="utf-8-sig")
    print(f"\n✅ 已保存汇总到: {out_file}")

    # 整体最早/最晚日期
    global_start = summary["start"].min()
    global_end = summary["end"].max()
    print(f"\n📅 整体最早日期: {global_start}")
    print(f"📅 整体最晚日期: {global_end}")

if __name__ == "__main__":
    main()


发现 parquet 文件数量: 5171

=== 每个标的的起止日期 & 行数（前 10 条） ===
                file       code      start        end  rows
0  000001.SZ.parquet  000001.SZ 2025-11-17 2025-12-05    15
1  000002.SZ.parquet  000002.SZ 2025-11-17 2025-12-05    15
2  000004.SZ.parquet  000004.SZ 2025-11-17 2025-12-05    15
3  000006.SZ.parquet  000006.SZ 2025-11-17 2025-12-05    15
4  000007.SZ.parquet  000007.SZ 2025-11-17 2025-12-05    15
5  000008.SZ.parquet  000008.SZ 2025-11-17 2025-12-05    15
6  000009.SZ.parquet  000009.SZ 2025-11-17 2025-12-05    15
7  000010.SZ.parquet  000010.SZ 2025-11-17 2025-12-05    15
8  000011.SZ.parquet  000011.SZ 2025-11-17 2025-12-05    15
9  000012.SZ.parquet  000012.SZ 2025-11-17 2025-12-05    15

✅ 已保存汇总到: /home/kunama/Personal/Stock/Code/trade/data_center/storage/market_data/stock_daily/stock_daily_datetime_range_summary.csv

📅 整体最早日期: 2025-11-14 00:00:00
📅 整体最晚日期: 2025-12-05 00:00:00
